# pandas cheat sheet — MATH4SDG 2026

**Executable quick reference.** You can run every cell independently of the insurance and temperature projects, working from top to bottom once to create the example tables. Later, return to the section you need. This is **not a compulsory exercise**.

Everything below is **invented classroom data**, not claims from WCF, TIRA, an insurer, or any Tanzanian business. The [insurance investigation](../notebooks/synthetic_insurance_claims.ipynb) uses different synthetic records. The [temperature investigation](../notebooks/east_africa_temperature_trends.ipynb) distinguishes NASA POWER *model-informed grid-cell estimates* from actual station observations.

A pandas **DataFrame** is a table of rows and columns. A **Series** is one column (or another one-dimensional sequence) with an index.

[Official pandas introduction](https://pandas.pydata.org/docs/user_guide/10min.html).


## 1. Create a tiny example table

These eight fictional establishment-year records are deliberately simple. The column `worker_years` represents **exposure**: 100 worker-years could mean 100 workers insured for one year or 200 workers insured for half a year.


In [ ]:
import pandas as pd

data = pd.DataFrame({
    "year": [2024, 2024, 2025, 2025, 2024, 2025, 2024, 2025],
    "sector": ["Transport", "Transport", "Services", None,
               "Services", "Transport", "Retail", "Retail"],
    "worker_years": [10., 90., 20., 50., 80., 30., 40., 55.],
    "claims_count": [1, 1, 0, 1, 2, 1, 0, 1],
})
data


## 2. Inspect the table

In JupyterLab the **last expression in a cell displays automatically**. You normally do not need `print(data.head())` or `display(data.head())`.


In [ ]:
data.head()


`shape` gives the number of rows and columns.


In [ ]:
data.shape


Column names and types are useful when you do not yet know the structure of a dataset.


In [ ]:
data.dtypes


Count missing entries in each column. Missing values may be represented as `NaN` or `pd.NA`.


In [ ]:
data.isna().sum()


See which years occur. `tolist()` converts NumPy values to ordinary Python values.


In [ ]:
sorted(data["year"].unique().tolist())


See how often each sector occurs, **including missing labels**.


In [ ]:
data["sector"].value_counts(dropna=False)


## 3. Select data

Single brackets select a **Series**; double brackets select a **DataFrame** with selected columns.


In [ ]:
data["claims_count"]


In [ ]:
data[["year", "claims_count"]]


Select only rows in 2025 and show two columns. `loc` selects by labels or Boolean conditions; `iloc` selects by integer position.


In [ ]:
data.loc[data["year"] == 2025, ["sector", "claims_count"]]


In [ ]:
data.iloc[:3]


## 4. Missing values and copies

`copy()` protects the original table. Replacing missing sector labels is an explicit analytical choice, **not** a way to recover an unknown original category.


In [ ]:
clean_data = data.copy()
clean_data["sector"] = clean_data["sector"].fillna("Unreported")
clean_data[["year", "sector"]].head()


Without assigning the result, `fillna()` normally leaves the original column unchanged. For other data, such as temperature observations, filling missing measurements can change a scientific conclusion; do not do it automatically.

To create a new calculated column, work on a copy and check the units.


In [ ]:
individual = data.copy()
individual["claims_per_100_worker_years"] = (
    100 * individual["claims_count"] / individual["worker_years"]
)
individual[["sector", "worker_years", "claims_count",
            "claims_per_100_worker_years"]].round(2)


Some numerical libraries expect an array rather than a pandas Series. `to_numpy()` extracts the values as a NumPy array.


In [ ]:
exposure = data["worker_years"].to_numpy()
exposure


## 5. Combine groups — calculate rates in the right order

**Do not average establishment-level rates without considering exposure.** First add the claim counts and worker-years within each sector; *then* divide the totals.

**Step 1:** Use the table with labelled missing sectors (created above).


In [ ]:
clean_data[["sector", "claims_count", "worker_years"]].head()


**Step 2:** Group by sector. Double brackets choose the two columns we want to sum.


In [ ]:
by_sector = clean_data.groupby("sector")[["claims_count", "worker_years"]].sum()
by_sector


**Step 3:** Calculate the frequency per 100 worker-years from the totals.


In [ ]:
by_sector["claims_per_100_worker_years"] = (
    100 * by_sector["claims_count"] / by_sector["worker_years"]
)
by_sector.round(2)


### Why this matters

The first two Transport records each contain **one claim**, but have **10** and **90** worker-years. Their respective rates are 10 and about 1.11 claims per 100 worker-years.

The combined rate is

\[
100\frac{1+1}{10+90}=2
\]

claims per 100 worker-years, **not** the unweighted mean, approximately 5.56. This is why we divide the sum of claims by the sum of exposures.


In [ ]:
transport = data.iloc[:2]
combined_rate = 100 * transport["claims_count"].sum() / transport["worker_years"].sum()
naive_average = (100 * transport["claims_count"] / transport["worker_years"]).mean()

pd.Series({
    "Combined rate per 100 worker-years": combined_rate,
    "Unweighted mean of establishment rates": naive_average,
}).round(2)


`groupby(...).mean()` can be useful for other questions; it simply answers a **different** question.

[Official pandas groupby guide](https://pandas.pydata.org/docs/user_guide/groupby.html).


## 6. Time series — checking dates and missing observations

For a **separate, fictional temperature example**, make a Series whose index contains dates. These are invented values, **not NASA POWER measurements**.


In [ ]:
dates = pd.to_datetime(["2025-01-30", "2025-01-31",
                        "2025-02-01", "2025-02-03", "2025-02-04"])
temperatures = pd.Series([28.0, 28.4, 29.2, 27.6, 28.1],
                         index=dates, name="invented_temperature_c")
temperatures


`asfreq("D")` makes the daily calendar explicit. Here February 2 has **no recorded value**, so pandas shows `NaN`. This is *not* the same as an observed temperature of zero.


In [ ]:
daily = temperatures.asfreq("D")
daily


`resample("MS")` groups values by **calendar month**, indexed by the month start. `mean()` ignores missing values when averaging.


In [ ]:
monthly = daily.resample("MS").mean()
monthly


A monthly mean based on very few days may be misleading. Count the observed days in each month **before** using the summaries scientifically.


In [ ]:
observed_days = daily.resample("MS").count()
observed_days


The temperature project uses a stricter **minimum valid days per month** rule when calculating annual summaries. It never treats a partial month as automatically equivalent to a complete one.

[Official pandas time-series guide](https://pandas.pydata.org/docs/user_guide/timeseries.html) · [Missing-data guide](https://pandas.pydata.org/docs/user_guide/missing_data.html).


## 7. Ask AI to explain a command — then verify

Useful focused questions:

> Explain `data.loc[data["year"] == 2025, ["sector", "claims_count"]]` using an invented three-row example. What does it return?

> Why is the total-claims-to-total-exposure ratio different from an unweighted average of individual rates? Give numbers I can check.

> Rewrite this pandas operation in two simple steps, and suggest a check that the result is unchanged.

**Check AI suggestions yourself:** inspect rows, missing values, units, aggregate totals and whether any observations were silently excluded. Never upload confidential partner records to an external AI service without permission.

### More official help

- [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html)
- [Group by: split–apply–combine](https://pandas.pydata.org/docs/user_guide/groupby.html)
- [Missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html)
- [Time series](https://pandas.pydata.org/docs/user_guide/timeseries.html)
- [Plotting](https://pandas.pydata.org/docs/user_guide/visualization.html)

*Educational explanations: CC BY 4.0. Code cells: MIT. The examples are synthetic and available for experimentation.*
